# Lesson 56 · Backpropagation by hand — sending the gradient swimming upstream: topological sort × the chain rule

**Goal**: first get the order of backpropagation clear, then implement `Value.backward()` — topological sort × chain rule, so the gradient really flows upstream.

> **One of the most important DL lessons**: first work through the topological order and the chain-rule multiplications by hand on 3 nodes, then write the code. Links back to L24.

🔗 **Aurora connection**: this is exactly what PyTorch's `.backward()` does; the Month 2 training loop (weight updates) depends on this mechanism.

← **Previous lesson** · [L55 · Completing the operators](L55_forward_pass.ipynb)

> Last lesson covered **completing the operators**: implementing the pow, relu, tanh and exp nodes, and the full computation graph.
> This lesson explores **backpropagation by hand**.

> **Difficulty bridge (L55 forward → L56 backward)**: last lesson you let the data "flow downstream", from the leaves to the output `L` (the forward pass); this lesson opens the sluice gate the other way and sends the gradient "swimming upstream", from `L` back to every parameter. The operators, the computation graph and the local-derivative formulas **have not changed at all** — the only thing that changes is **the direction of travel**. So don't worry: the only genuinely new thing is "in what order to walk backwards".

## This lesson's story: get the order wrong, and the gradient is wrong

The last two lessons gave every operator a "local backward function". But one problem is still unsolved: **in what order do we call these functions?**

A bad example: calling `_backward()` in random order. Take the computation graph `L = a*b + c`:
- if we call the `_backward()` of `a*b` first, `(a*b).grad` is still 0 (the `+` hasn't passed anything back yet) → `a.grad` and `b.grad` both come out as 0

**Conclusion: we must go backwards from the output node towards the input nodes, and each node has to wait until every node after it has been handled before its own turn comes.** This is the DAG's **reverse topological order**.

The algorithm takes just three steps:

```
1. Starting from L, do a DFS post-order traversal of the DAG → gives the topological sequence topo
2. L.grad = 1.0                  (dL/dL = 1, the starting point of the chain rule)
3. for node in reversed(topo):   → call each node's _backward() in reverse order
       node._backward()
```

**The gradient-accumulation trap**: when the same node `a` is used by two operators (`L = a*a`), gradients arrive along two paths and must accumulate with `+=`, not be overwritten with `=`.

Task for this section: implement `Value.backward()`, so that `L.backward()` backpropagates through the whole graph in one call.

## 🤔 Why did engineers invent this?

- **What happens without it?** For a network with a million parameters, you would have to write out a partial-derivative formula for every parameter by hand and then evaluate each one — you'd never finish writing them, and you could never debug them.
- **What real problem does it solve?** Backpropagation turns "find the gradients of all the parameters" into **a single graph traversal**: it reuses the intermediate values already computed in the forward pass, and one `for` loop fills in every `dL/d(parameter)`, at a cost about the same as one forward pass.
- **Where does it come up again?** The whole-network backward pass of the L57 MLP, the `w -= lr*w.grad` of the L58 training loop, PyTorch's `.backward()` in L60 — all of them are scaled-up versions of this same mechanism.

## ⚠️ Common misconceptions

> Don't think of backprop as some "mysterious learning algorithm". It is **just the chain rule** — `dL/dx = dL/dg · dg/dx` unrolled **mechanically, in reverse topological order,** along the computation graph. There is no magic in it, and no "learning" either. What actually makes the network learn is the gradient descent that uses these gradients afterwards (L58).

In [ ]:
import numpy as np

# ── Basic Value implementation (with each operation's _backward; backward itself is left for below) ──
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _bwd():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _bwd
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _bwd():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _bwd
        return out

    def __neg__(self):        return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')
        def _bwd():
            self.grad += (out.data > 0) * out.grad
        out._backward = _bwd
        return out

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

print("Value class loaded")

## 💧 Intuition: think of the gradient as water flowing through pipes

Before the formal derivation, let's build a picture. Think of the computation graph as a **network of water pipes**, with the gradient as the water flowing through it:

- The output node `L` is the **main water source**, pumping back a fixed amount of 1 (this is the `L.grad = 1.0` you'll see shortly).
- Each node receives an **upstream flow** (`out.grad`), multiplies it by its own pipe section's **local derivative** (the local partial derivative) to get the flow for each input, and passes that on **downstream** (towards the inputs).
  - Example: for `out = a * b`, the flow `a` receives = upstream flow × `b` (the local derivative of `a`'s pipe section is exactly `b`).
- If **two pipes flow into** one node (it's reused, as in `L = a*a`), the two streams **add up** — that's why we accumulate with `+=` instead of overwriting.
- **A node must wait until all the water from upstream has arrived before it splits it onwards** — that is exactly the "reverse topological order" coming up below.

Remember this one line: "**upstream flow × local derivative, then pass it downstream**" — that is the whole of backpropagation. Below we turn this picture into a rigorous algorithm.

### A note on notation: `d` vs `∂` (total vs partial derivative)

When deriving backpropagation you'll see two notations: `dL/dx` and `∂L/∂x`. What's the difference?

- **`d` denotes a total derivative**: used for the derivative when one variable affects another along **only one path**
- **`∂` denotes a partial derivative**: used for a multivariable function, differentiating with respect to one variable while holding the others fixed

**Example**: the computation graph `L = a*b + c`
- `a` affects `L` along only one path (`a → a*b → L`), so writing `dL/da` is unambiguous
- for a function `f(a, b, c) = a*b + c`, the derivative with respect to `a` is usually written `∂f/∂a` (meaning `b, c` are held fixed)

**In backpropagation practice**:
- we use `dL/dx` or `∂L/∂x` to mean "the partial derivative of the loss `L` with respect to the parameter `x`"
- in the context of backpropagation the two notations are usually interchangeable, because the computation graph automatically keeps track of every path from each parameter to the output
- mathematically the more rigorous notation is `∂L/∂x`, but in deep-learning code the shorthand `dL/dx` is common

**What matters is understanding the meaning**: `dL/dx` means "if `x` increases by 1 unit, how much does `L` change", whatever symbol you use.

## Concept 1: the topological sort guarantees the order of gradient accumulation

A computation graph is a DAG (directed acyclic graph). A node `v`'s `_backward()` passes `v.grad` on to its inputs, and `v.grad` itself comes from the upstream gradients sent back by every node "that used `v.data`" — those upstream gradients must all have accumulated first.

A topological sort gives a linear order such that: if `u → v` (`u` depends on `v`, i.e. `v` is an input of `u`), then `u` comes after `v`.
Walking this order in reverse guarantees that whenever a node's `_backward()` is called, its `.grad` has already collected every upstream gradient coming from the output side.

A DFS post-order traversal naturally produces a topological order: first recursively visit all the child nodes, and only after that add the current node to the list.

```
topo = []
visited = set()
def build(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:   # walk towards the inputs
            build(child)
        topo.append(v)          # post-order: add yourself last
build(output)
# topo[-1] == output, topo[0] is a leaf node
```

### Which way do the arrows point? Forward data flow vs topological dependency

This is easy to mix up. Let's pin down the direction of two kinds of "arrow":

**Arrows in the forward computation graph** (how data flows):
- input → operator → output
- e.g. in `c = a + b` the arrows point to `c` (data flows from `a, b` to `c`)

**Dependency arrows in the topological sort** (who depends on whom):
- here `u → v` means "`u` depends on `v`" — i.e. `v` is an input of `u`
- the topological sort requires: if `u` depends on `v`, then `u` comes **after** `v`
- **this means the arrows in the topological sort are reversed!** They point towards the inputs

**In other words**: during backpropagation we trace back along the `_prev` pointers **towards the inputs**, not in the direction of the forward arrows. That's why we write `for child in v._prev: build(child)` (walking backwards) and not `for out in v._next:` (walking forwards).

In [ ]:
# Demo of the topological order
a = Value(2.0); b = Value(3.0); c = Value(-1.0)
L = a * b + c   # L = a*b + c

topo = []
visited = set()
def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

build_topo(L)
print("Topological order (from the leaves to the output):")
for i, v in enumerate(topo):
    print(f"  [{i}] op={v._op!r:5s}  data={v.data:.2f}  prev_count={len(v._prev)}")
print()
print("Reversed (the direction backward walks):", [v._op or 'leaf' for v in reversed(topo)])

### Why does a post-order traversal give the right topological order? (A counter-example)

What happens if we add nodes in **pre-order** instead (first `topo.append(v)`, then recurse)?

```
def build_preorder(v):   # ❌ pre-order (wrong)
    if v not in visited:
        visited.add(v)
        topo.append(v)        # ← add yourself first
        for child in v._prev:
            build_preorder(child)
```

The `topo` you get this way comes out **back to front** — the output node is added first and the leaf nodes last. When we flip it with `reversed(topo)`, the output node ends up last, which is exactly the wrong way round.

The concrete effect: walking in reverse calls `_backward()` on the **input nodes** first, but their gradients haven't arrived from the **output** yet (because the output hasn't been handled), so everything they accumulate is 0.

**Proof: try a pre-order traversal**

In [ ]:
# Pre-order vs post-order
a_demo = Value(2.0); b_demo = Value(3.0); c_demo = Value(-1.0)
L_demo = a_demo * b_demo + c_demo

# Post-order traversal (correct)
topo_post = []
visited_post = set()
def build_postorder(v):
    if v not in visited_post:
        visited_post.add(v)
        for child in v._prev:
            build_postorder(child)
        topo_post.append(v)
build_postorder(L_demo)

# Pre-order traversal (wrong)
topo_pre = []
visited_pre = set()
def build_preorder(v):
    if v not in visited_pre:
        visited_pre.add(v)
        topo_pre.append(v)        # ← added first
        for child in v._prev:
            build_preorder(child)
build_preorder(L_demo)

print("Post-order traversal (correct):", [v._op or 'leaf' for v in topo_post])
print("  reversed (backward walk):", [v._op or 'leaf' for v in reversed(topo_post)])
print()
print("Pre-order traversal (wrong):", [v._op or 'leaf' for v in topo_pre])
print("  reversed (backward walk):", [v._op or 'leaf' for v in reversed(topo_pre)])
print()
print("Conclusion: reversing the pre-order walks from the leaves to the output, which puts the gradient computations in the wrong order")

### The recursion in detail: the full DFS trace for `L = a*b + c`

Let's trace by hand how `build_topo()` handles `L = a*b + c`. The structure of the computation graph:

```
a ──┐
    ├─→ (a*b) ──┐
b ──┘           ├─→ L (output)
                │
c ──────────────┘
```

**The order of recursive calls** (the brackets show the calls on the stack; `_prev` is a set, so the order in which children are visited isn't fixed — the table follows one order, with mul before c and a before b, and any order gives a valid topological order):

| Step | Recursion stack | Action | Current contents of topo |
|------|--------|------|-------------|
| 1 | `build(L)` | L not visited; mark L visited | `[]` |
| 2 | `build(L) → build(mul_node)` | mul_node not visited; mark visited | `[]` |
| 3 | `build(mul) → build(a)` | a not visited; mark visited; a has no children | `[]` |
| 4 | `build(mul) ← a returns` | a has no more children, **add a in post-order** | `[a]` |
| 5 | `build(mul) → build(b)` | b not visited; mark visited; b has no children | `[a]` |
| 6 | `build(mul) ← b returns` | b has no more children, **add b in post-order** | `[a, b]` |
| 7 | `build(mul) ← returns` | mul has handled all its children, **add mul in post-order** | `[a, b, mul]` |
| 8 | `build(L) → build(c)` | c not visited; mark visited; c has no children | `[a, b, mul]` |
| 9 | `build(L) ← c returns` | c has no more children, **add c in post-order** | `[a, b, mul, c]` |
| 10 | `build(L) ← returns` | L has handled all its children, **add L in post-order** | `[a, b, mul, c, L]` |

**Final result**: `topo = [a, b, mul, c, L]`
**Reversed (the backward walk)**: `reversed(topo) = [L, c, mul, b, a]`

**Why this order is right**:
- first L's `_backward()` is called (it runs automatically after L.grad = 1.0 is initialized)
- then c's `_backward()` (c appears only among L's inputs; its gradient comes from L)
- then mul's `_backward()` (mul also appears only among L's inputs)
- only last are a's and b's `_backward()` called (they appear among mul's inputs; their gradients come from mul)

This way, when each node is handled, its gradient has already finished accumulating from every node "downstream that uses it".

## Concept 2: start from the output node with `self.grad = 1.0`

The gradient of the loss `L` with respect to itself is always 1: `dL/dL = 1`.
This is the starting point of the chain rule — every later `_backward()` takes the "upstream gradient" (i.e. `out.grad`), multiplies it by a local derivative, and writes the result into an input node's `.grad`.

```
self.grad = 1.0          # dL/dL = 1
for node in reversed(topo):
    node._backward()     # chain-rule propagation
```

If you forget to set `self.grad = 1.0`, every term multiplied by `out.grad` becomes 0, and all the gradients are zero.

In [ ]:
# Demo: without grad=1.0, every gradient is zero
a2 = Value(2.0); b2 = Value(3.0)
out2 = a2 * b2
# call _backward by hand, but without setting out2.grad
out2._backward()
print("Without out.grad=1.0: a2.grad =", a2.grad, "(expected 3.0, but got 0)")

# The right way
a3 = Value(2.0); b3 = Value(3.0)
out3 = a3 * b3
out3.grad = 1.0          # ← the key line
out3._backward()
print("With out.grad=1.0: a3.grad =", a3.grad, "(expected 3.0) ✅")

## Concept 3: accumulate gradients (`+=`), don't overwrite them (`=`)

When the same node is used by several later nodes (a diamond graph), gradients come in along several paths and must be **added up**; a later path must not overwrite an earlier one.

Example: `L = a*a` (`a` is both the left and the right operand)
- Path 1: `dL/d(left a) = a.data = 2`
- Path 2: `dL/d(right a) = a.data = 2`
- Total gradient: `a.grad = 2 + 2 = 4` (equal to the closed form `dL/da = 2a = 4`)

Writing `self.grad += ...` everywhere in `_backward`, rather than `self.grad = ...`, takes care of diamond graphs automatically.

In [ ]:
# Diamond-graph demo: L = a * a
a4 = Value(2.0)
L4 = a4 * a4          # a4 is both operands (the same object; since _prev is a set, it holds a4 only once)
L4.grad = 1.0
L4._backward()         # run mul's _backward once
print(f"a4.grad = {a4.grad:.1f}  (expected 4.0, because d(a^2)/da = 2a = 4)")
# Note: mul's _backward uses +=, so both paths get accumulated

### A closer look: why does `+=` accumulation match adding up the paths when differentiating?

This is no coincidence; it is **a property of the chain rule itself**.

**The chain rule along a single path**:
$$\frac{dL}{da} = \frac{dL}{d(\text{out})} \cdot \frac{d(\text{out})}{da}$$

**The multi-path case** (e.g. `L = a*a`):
- Path 1: the contribution of `L → (the first a)` is $\frac{dL}{d(\text{mul})} \cdot \frac{d(\text{mul})}{d(\text{left_a})} = 1 \cdot a = a$
- Path 2: the contribution of `L → (the second a)` is $\frac{dL}{d(\text{mul})} \cdot \frac{d(\text{mul})}{d(\text{right_a})} = 1 \cdot a = a$
- **Total gradient**: add the two paths: $a + a = 2a$

In code, these two paths correspond to the two update lines in `mul`'s `_backward()`:
```python
def _bwd():
    self.grad  += other.data * out.grad    # ← path 1: left_a receives other.data * out.grad
    other.grad += self.data  * out.grad    # ← path 2: right_a receives self.data * out.grad
```

When `self` and `other` are the same object (namely `a`), both lines accumulate into the same memory location, `a.grad`, which automatically adds up the paths.

**That's why you must use `+=` and not `=`**: `=` would let the later path's contribution overwrite the earlier one, and the gradient would come out wrong.

## 1. ✏️ Implement `Value.backward(self)`

**A three-step implementation route**:

| Step | Action | Key point |
|---|---|---|
| 1 | DFS post-order traversal from `self` → the `topo` list | use a `visited` set to avoid repeats; walk `node._prev` |
| 2 | `self.grad = 1.0` | dL/dL=1, the starting point of every gradient computation |
| 3 | `for node in reversed(topo): node._backward()` | the reverse order guarantees "upstream finishes first" |

**Acceptance criteria**:
```python
a, b, c = Value(2.0), Value(3.0), Value(-1.0)
L = a * b + c    # L.data = 5.0
L.backward()
assert abs(a.grad - 3.0) < 1e-12   # dL/da = b = 3
assert abs(b.grad - 2.0) < 1e-12   # dL/db = a = 2
assert abs(c.grad - 1.0) < 1e-12   # dL/dc = 1
```

**Diamond-graph check** (`L = a*a`, `a=2`):
- `dL/da` comes from two paths: left operand (`a.data=2`) + right operand (`a.data=2`) → `grad = 4.0`
- if you mistakenly use `=` instead of `+=`, only the last path is kept and you get `2.0` (wrong)

In [ ]:
def backward(self):
    # ✏️ TODO step 1: DFS post-order traversal, building the topological sequence topo
    topo = []
    visited = set()
    def build(v):
        raise NotImplementedError("TODO: recursively visit v._prev and add v to topo in post-order")

    build(self)

    # ✏️ TODO step 2: set the output node's gradient to 1.0
    # TODO: self.grad = 1.0

    # ✏️ TODO step 3: call each node's _backward() in reverse order
    pass

# Attach the method to the Value class
Value.backward = backward  # replace it directly

In [ ]:
# ⚠️ Run this cell only after you've implemented backward() in the exercise cell above. If it isn't implemented, you get a note instead of a crash.
try:
    # Check: L = a*b + c
    a = Value(2.0); b = Value(3.0); c = Value(-1.0)
    L = a * b + c
    L.backward()

    assert abs(a.grad - 3.0) < 1e-9, f"a.grad should be 3.0, got {a.grad}"
    assert abs(b.grad - 2.0) < 1e-9, f"b.grad should be 2.0, got {b.grad}"
    assert abs(c.grad - 1.0) < 1e-9, f"c.grad should be 1.0, got {c.grad}"
    print(f"a.grad={a.grad:.1f}  b.grad={b.grad:.1f}  c.grad={c.grad:.1f}")
    print("✅ backward() basic check passed")

    # Diamond-graph check: L = a*a, dL/da = 2*a = 4
    a2 = Value(2.0)
    L2 = a2 * a2
    L2.backward()
    assert abs(a2.grad - 4.0) < 1e-9, f"a2.grad should be 4.0, got {a2.grad}"
    print(f"Diamond graph a2.grad={a2.grad:.1f} (expected 4.0)")
    print("✅ Diamond-graph gradient-accumulation check passed")
except (NotImplementedError, TypeError):
    print('⬜ TODO: implement backward() in the exercise cell above first, then run this cell to check it.')
except AssertionError as e:
    print(f'❌ Check failed: {e}')
    print('💡 Check the direction of your topo order and your grad accumulation logic.')


## Parameter experiment: checking the gradient with numerical differentiation

Use numerical differentiation, `(f(x+h) - f(x-h)) / (2h)`, to compute the gradient independently and compare it with the result of `backward()`; the gap should be `< 1e-4`.

Experiment settings:
- `h = 1e-5` (too large a step gives a big error; too small and floating-point jitter takes over)
- test function: `L = a*b + c.relu()` (a mixed expression including relu)
- what to expect: `a.grad` equals `b.data`, and the gap between numerical differentiation and the analytic gradient is `< 1e-4`

## How do we check a gradient is right? An introduction to numerical differentiation

### Why numerical differentiation?

Once `backward()` is implemented, we need to check **independently** whether its gradients are right. Numerical differentiation gives a reference value that doesn't depend on the code's logic.

### The textbook definition of the derivative, and its problem

The definition of the derivative you learned at school:
$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

But a computer can't take a limit. We approximate with a finite $h$:
$$f'(x) \approx \frac{f(x+h) - f(x)}{h} \quad (\text{one-sided difference})$$

**The problem**: this approximation has a large error. For example, for $f(x) = x^2$ at $x=2$ with $h=0.001$, the approximation is $\frac{(2.001)^2 - 2^2}{0.001} = 4.001$, while the exact value is $4.0$ — off by $0.001$.

### The central difference: a more accurate formula

Use **two points placed symmetrically**:
$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} \quad (\text{central difference})$$

**Why is it more accurate?** By Taylor expansion (you don't need to follow it fully; just look at the conclusion):
$$f(x+h) \approx f(x) + f'(x) \cdot h + \frac{f''(x)}{2} \cdot h^2 + O(h^3)$$
$$f(x-h) \approx f(x) - f'(x) \cdot h + \frac{f''(x)}{2} \cdot h^2 + O(h^3)$$

Subtract: $f(x+h) - f(x-h) \approx 2f'(x) \cdot h + O(h^3)$

So the error of the central difference is $O(h^2)$ (second order), while the one-sided difference is $O(h)$ (first order). **The central difference is accurate to a whole extra order in $h$** — with $h = 10^{-5}$ that is a huge gain.

### Choosing the step h: why 1e-5?

- **Too large** ($h = 0.1$): big approximation error, the numerical gradient is inaccurate
- **Too small** ($h = 1e-10$): floating-point rounding error dominates, so it gets *less* accurate
- **Sweet spot**: usually somewhere between $h = 10^{-4}$ and $10^{-6}$

In this range both the higher-order Taylor terms and the floating-point rounding error stay relatively small. $h = 10^{-5}$ is an empirical compromise, and many deep-learning frameworks use it.

**The idea of the check**: compute the gradient from `backward()` and the numerical gradient; if the error is `< 1e-4`, the `backward()` implementation is correct.

In [ ]:
def numeric_grad(f, x, h=1e-5):
    """Numerical gradient of f at scalar input x (central difference)."""
    return (f(x + h) - f(x - h)) / (2 * h)

# hold the other parameters fixed and nudge one input at a time
a_val, b_val, c_val = 2.0, 3.0, -0.5

def L_a(av): return (Value(av) * Value(b_val) + Value(c_val).relu()).data
def L_b(bv): return (Value(a_val) * Value(bv) + Value(c_val).relu()).data
def L_c(cv): return (Value(a_val) * Value(b_val) + Value(cv).relu()).data

num_a = numeric_grad(L_a, a_val)
num_b = numeric_grad(L_b, b_val)
num_c = numeric_grad(L_c, c_val)

# Analytic gradient (backward) — depends on your implementation in the exercise cell above
try:
    a = Value(a_val); b = Value(b_val); c = Value(c_val)
    L = a * b + c.relu()
    L.backward()

    print(f"{'var':<6} {'backward':>12} {'numerical':>12} {'error':>12}")
    for name, ana, num in [('a', a.grad, num_a), ('b', b.grad, num_b), ('c', c.grad, num_c)]:
        err = abs(ana - num)
        flag = "✅" if err < 1e-4 else "❌"
        print(f"{name:<6} {ana:>12.6f} {num:>12.6f} {err:>12.2e} {flag}")
except (NotImplementedError, TypeError):
    print(f"Numerical-differentiation reference values: num_a={num_a:.6f}, num_b={num_b:.6f}, num_c={num_c:.6f}")
    print('⬜ TODO: after implementing backward(), rerun this cell to compare the analytic gradient with numerical differentiation.')

## 🎯 The payoff later

The **backpropagation** you worked out yourself today comes back in **L57 / L58 / L60** — L57 uses it to run the backward pass through a whole MLP, L58 relies on it to update the weights in the training loop (`w.data -= lr * w.grad`), and by L60 you'll find that PyTorch's `loss.backward()` does exactly the graph traversal you wrote by hand today. Once you've written it yourself, whenever you call autograd you'll always know what's happening under the hood.

## Wrap-up

`Value.backward()` builds a topological order with a DFS post-order traversal, then calls each node's `_backward()` in reverse, so that every leaf node's `.grad` collects all the partial derivatives coming from the output side.
Gradients accumulate with `+=`, which guarantees that in a diamond graph (a reused parameter) the contributions of the different paths add up correctly.
This mechanism maps straight onto the parameter-update step of the training loop later on (see **L58**, `L58_training_loop.ipynb`): `w.data -= lr * w.grad` — and the training loop of the Aurora ML engine (`src/aurora/ml/`, a planned module) follows the same pattern.
The next lesson (**L57**) builds on this to assemble `Neuron → Layer → MLP`, wiring many `Value` nodes into a real network and running the whole network's forward + backward pass; the complete training loop with parameter updates waits until L58.

## ✏️ Closed-book derivation check cell — the matrix formulas of backpropagation

**Rule: close every cell above; first write out the derivation in this Markdown cell, then implement it in the code cell below.**

**Problem**: the forward pass of a 2-layer MLP:
$$z^{(1)} = W^{(1)} x + b^{(1)}, \quad a^{(1)} = \sigma(z^{(1)})$$
$$\hat{y} = W^{(2)} a^{(1)} + b^{(2)}, \quad L = \tfrac{1}{2}(\hat{y} - y)^2$$

Derive (write it here, without looking at your notes):

1. $\dfrac{\partial L}{\partial W^{(2)}} = ?$ (in outer-product form)

2. $\delta^{(1)} = \dfrac{\partial L}{\partial z^{(1)}} = ?$ (a Hadamard product involving $\sigma'(z^{(1)})$)

3. $\dfrac{\partial L}{\partial W^{(1)}} = ?$ (in outer-product form)

(Write your derivation here...)

## The mask in the ReLU gradient, in detail

### ReLU's mathematical definition and derivative

The ReLU function is defined as:
$$\text{ReLU}(z) = \max(0, z) = \begin{cases} z & \text{if } z > 0 \\ 0 & \text{otherwise} \end{cases}$$

Its derivative is a piecewise function:
$$\text{ReLU}'(z) = \frac{d}{dz} \text{ReLU}(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \le 0 \end{cases}$$

**Key observation**: the derivative works like a "switch" — only at positions where the activation is switched on (`z > 0`) can the gradient pass through; at every other position the gradient is cut to 0.

### Why is it a mask?

When $z$ is a vector, the derivative of ReLU is a vector too, with a 0 or a 1 at each position:
$$\text{ReLU}'(z) = \begin{pmatrix} \mathbb{1}_{z_1 > 0} \\ \mathbb{1}_{z_2 > 0} \\ \vdots \\ \mathbb{1}_{z_n > 0} \end{pmatrix} = \begin{pmatrix} 1 \text{ or } 0 \\ 1 \text{ or } 0 \\ \vdots \end{pmatrix}$$

Here $\mathbb{1}_{z_i > 0}$ is the **indicator function**: 1 when the condition is true, 0 otherwise.

This 0/1 vector is called a **mask**, because during backpropagation it "masks out" some of the gradient components.

### How it's used in a neural network's backward pass

Consider the backward pass of a two-layer network:
$$\frac{\partial L}{\partial z^{(1)}} = \frac{\partial L}{\partial a^{(1)}} \odot \text{ReLU}'(z^{(1)})$$

where $\odot$ is the Hadamard product (elementwise multiplication).

**A numerical example**: suppose
$$z^{(1)} = \begin{pmatrix} 0.5 \\ -0.3 \\ 1.2 \\ -0.1 \end{pmatrix}, \quad \frac{\partial L}{\partial a^{(1)}} = \begin{pmatrix} 0.1 \\ 0.2 \\ 0.15 \\ 0.05 \end{pmatrix}$$

Then the mask is:
$$\text{ReLU}'(z^{(1)}) = (z^{(1)} > 0) = \begin{pmatrix} 1 \\ 0 \\ 1 \\ 0 \end{pmatrix}$$

And the gradient is:
$$\frac{\partial L}{\partial z^{(1)}} = \begin{pmatrix} 0.1 \\ 0.2 \\ 0.15 \\ 0.05 \end{pmatrix} \odot \begin{pmatrix} 1 \\ 0 \\ 1 \\ 0 \end{pmatrix} = \begin{pmatrix} 0.1 \\ 0 \\ 0.15 \\ 0 \end{pmatrix}$$

**Observation**: the gradients at positions 2 and 4 (where $z < 0$) have been **zeroed out**, because ReLU's derivative is 0 there.

### From the mask to the weight gradient

Next, to find $\frac{\partial L}{\partial W^{(1)}}$, we keep applying the chain rule:
$$\frac{\partial L}{\partial W^{(1)}} = \frac{\partial L}{\partial z^{(1)}} \otimes x^T = \begin{pmatrix} 0.1 \\ 0 \\ 0.15 \\ 0 \end{pmatrix} \otimes \begin{pmatrix} x_1 & x_2 & \cdots \end{pmatrix}$$

So ReLU's mask acts **indirectly** on the weight gradient too: the weight rows belonging to activations that ReLU masked out don't get updated either (their gradient is 0).

## Matrix calculus basics: the outer product and the Hadamard product

So far we've only dealt with scalars. But neural networks involve matrices and vectors, and their gradients are matrices too. This section adds two key ideas.

### The outer product

**Definition**: given a column vector $u$ (shape $m \times 1$) and a row vector $v^T$ (shape $1 \times n$), the outer product is defined as:
$$u \otimes v^T = u v^T = \begin{pmatrix} u_1 \\ u_2 \\ \vdots \\ u_m \end{pmatrix} \begin{pmatrix} v_1 & v_2 & \cdots & v_n \end{pmatrix} = \begin{pmatrix} u_1 v_1 & u_1 v_2 & \cdots & u_1 v_n \\ u_2 v_1 & u_2 v_2 & \cdots & u_2 v_n \\ \vdots & \vdots & \ddots & \vdots \\ u_m v_1 & u_m v_2 & \cdots & u_m v_n \end{pmatrix}$$

**Result**: an $m \times n$ matrix whose $(i,j)$ entry is $u_i \cdot v_j$.

**Example**:
$$u = \begin{pmatrix} 2 \\ 3 \end{pmatrix}, \quad v^T = \begin{pmatrix} 1 & 4 \end{pmatrix} \quad \Rightarrow \quad u v^T = \begin{pmatrix} 2 \cdot 1 & 2 \cdot 4 \\ 3 \cdot 1 & 3 \cdot 4 \end{pmatrix} = \begin{pmatrix} 2 & 8 \\ 3 & 12 \end{pmatrix}$$

### Why is the gradient an outer product? Looking at the shapes

Consider the matrix product $\hat{y} = W \cdot a$, where:
- $W$ is a $1 \times 4$ matrix (the weights, whose gradient we want)
- $a$ is a $4 \times 1$ column vector (the activations)
- $\hat{y}$ is a $1 \times 1$ scalar (the output)

The gradient of the loss $L$ with respect to $W$ should have the same shape as $W$ (both $1 \times 4$).

From the chain rule:
$$\frac{\partial L}{\partial W_{ij}} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial W_{ij}}$$

Notice that $\hat{y} = \sum_k W_{ik} a_k$, so $\frac{\partial \hat{y}}{\partial W_{ij}} = a_j$.

Therefore:
$$\frac{\partial L}{\partial W_{ij}} = \frac{\partial L}{\partial \hat{y}} \cdot a_j$$

In matrix form:
$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial \hat{y}} \otimes a^T$$

That is, the outer product of "the upstream gradient" with "the transposed activations". This guarantees the shapes match: $(1 \times 1) \otimes (4 \times 1)^T = (1 \times 1) \otimes (1 \times 4) = (1 \times 4)$ ✓

### The Hadamard product (elementwise multiplication)

**Definition**: for two matrices of the same shape, multiply the entries in matching positions:
$$A \odot B = \begin{pmatrix} a_{11} b_{11} & a_{12} b_{12} & \cdots \\ a_{21} b_{21} & a_{22} b_{22} & \cdots \\ \vdots & \vdots & \ddots \end{pmatrix}$$

**Notation**: in maths it's written $A \odot B$; in NumPy it's `*` (on arrays).

**Example**:
$$A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}, \quad B = \begin{pmatrix} 5 & 6 \\ 7 & 8 \end{pmatrix} \quad \Rightarrow \quad A \odot B = \begin{pmatrix} 5 & 12 \\ 21 & 32 \end{pmatrix}$$

**How it differs from the matrix product `@`**:
- Hadamard product `*`: elementwise multiplication; the result has the same shape as the inputs
- matrix product `@`: linear combinations of rows × columns; the result usually has a different shape

### How it's used in backpropagation

When the activation function is nonlinear (such as ReLU or sigmoid), its derivative acts **elementwise**:
$$\text{ReLU}'(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{otherwise} \end{cases}$$

In vector form, if $z$ is a vector, then:
$$\sigma'(z) = \begin{pmatrix} \sigma'(z_1) \\ \sigma'(z_2) \\ \vdots \end{pmatrix}$$

When the chain rule propagates, the gradient and the derivative are combined with a Hadamard product:
$$\frac{\partial L}{\partial z} = \frac{\partial L}{\partial a} \odot \sigma'(z)$$

This way the gradient at each position is scaled by the derivative at that same position (0 or 1 for ReLU, $\sigma(z)(1-\sigma(z))$ for sigmoid, and so on).

In [ ]:
# A concrete demo of the outer product and the Hadamard product
import numpy as np

print("=" * 60)
print("Outer product demo")
print("=" * 60)
u = np.array([[2], [3]])  # (2, 1) column vector
v_T = np.array([[1, 4]])  # (1, 2) row vector
outer_product = u @ v_T   # outer product
print(f"u = {u.T}  (shape {u.shape})")
print(f"v^T = {v_T}  (shape {v_T.shape})")
print(f"u ⊗ v^T = u @ v^T =\n{outer_product}  (shape {outer_product.shape})")
print()

print("=" * 60)
print("Hadamard product demo")
print("=" * 60)
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
hadamard = A * B  # NumPy's * on arrays is the Hadamard product
print(f"A =\n{A}")
print(f"B =\n{B}")
print(f"A ⊙ B (elementwise product) =\n{hadamard}")
print()

print("=" * 60)
print("A real example: the outer-product form of a neural-network gradient")
print("=" * 60)
# Suppose:
# - W2 is a (1, 4) weight matrix
# - a1 is a (4, 1) activation vector
# - dL/dyhat is a (1, 1) scalar gradient (or think of it as 1-D)
W2 = np.array([[0.1, 0.2, 0.3, 0.4]])  # (1, 4)
a1 = np.array([[1], [2], [3], [4]])    # (4, 1)
dL_dyhat = np.array([[0.5]])           # (1, 1) gradient of the scalar loss w.r.t. the output

# the gradient, computed as an outer product
dL_dW2 = dL_dyhat @ a1.T  # (1, 1) @ (4, 1) -> (1, 4) ✓
print(f"dL/dyhat = {dL_dyhat.T}  (shape {dL_dyhat.shape})")
print(f"a1 = {a1.T}  (shape {a1.shape})")
print(f"dL/dW2 = dL/dyhat ⊗ a1^T =")
print(f"{dL_dW2}  (shape {dL_dW2.shape})")
print()

print("Check: each entry of dL/dW2")
for i in range(W2.shape[0]):
    for j in range(W2.shape[1]):
        elem = dL_dyhat[i, 0] * a1[j, 0]
        print(f"  dL/dW2[{i},{j}] = dL/dyhat × a1[{j}] = {dL_dyhat[i,0]} × {a1[j,0]} = {elem}")
print()

print("=" * 60)
print("The Hadamard product in ReLU")
print("=" * 60)
z1 = np.array([[0.5], [-0.3], [1.2], [-0.1]])  # (4, 1)
mask = (z1 > 0).astype(float)  # ReLU derivative (used as a mask)
dL_da1 = np.array([[0.1], [0.2], [0.15], [0.05]])  # (4, 1)
dL_dz1 = dL_da1 * mask  # Hadamard product
print(f"z1 = {z1.T}")
print(f"mask (z1 > 0) = {mask.T}")
print(f"dL/da1 = {dL_da1.T}")
print(f"dL/dz1 = dL/da1 ⊙ mask(z1) = {dL_dz1.T}")
print("(note: the gradients at the negative positions were zeroed by the mask, because ReLU's derivative is 0 there)")

In [ ]:
# Check: implement the formulas you derived in this code cell and compare them with the numerical gradient
# You only need to fill in the three lines marked TODO; the rest of the code is given
import numpy as np

def sigmoid(z): return 1 / (1 + np.exp(-z))
def dsigmoid(z): s = sigmoid(z); return s * (1 - s)

np.random.seed(7)
W1 = np.random.randn(4, 3) * 0.1
b1 = np.zeros((4, 1))
W2 = np.random.randn(1, 4) * 0.1
b2 = np.zeros((1, 1))
x  = np.random.randn(3, 1)
y  = np.array([[1.0]])

# Forward pass
z1 = W1 @ x + b1; a1 = sigmoid(z1)
z2 = W2 @ a1 + b2; y_hat = z2
loss = 0.5 * (y_hat - y) ** 2

# ── Backward pass (replace the three zeros placeholders with your derivation) ────────────────────
dL_dyhat = y_hat - y
# ✏️ TODO 1: dL/dW2  (same shape as W2: the outer product of dL_dyhat and a1)
dL_dW2 = np.zeros_like(W2)        # ← replace with your derivation

# ✏️ TODO 2: δ¹ = dL/dz¹  (same shape as z1: pass back through W2 transposed, then Hadamard product with σ'(z1))
delta1 = np.zeros_like(z1)        # ← replace with your derivation

# ✏️ TODO 3: dL/dW1  (same shape as W1: the outer product of delta1 and x)
dL_dW1 = np.zeros_like(W1)        # ← replace with your derivation

# Numerical gradient check (finite differences) — reference values, don't change
def loss_fn(W2_, W1_):
    a1_ = sigmoid(W1_ @ x + b1)
    yh  = W2_ @ a1_ + b2
    return 0.5 * ((yh - y) ** 2).item()   # .item(): (1,1) array → scalar (float() raises an error under numpy 2.x)

eps = 1e-5
def num_grad(param, i, j, loss_fn_partial):
    p = param.copy(); p[i,j] += eps
    return (loss_fn_partial(p) - loss_fn_partial(param.copy())) / eps

if not (dL_dW2.any() or delta1.any() or dL_dW1.any()):
    print('⬜ TODO: first replace the three zeros placeholders with your hand-derived formulas, then run this cell for the numerical check.')
else:
    ng_W2_00 = num_grad(W2, 0, 0, lambda w: loss_fn(w, W1))
    ng_W1_00 = num_grad(W1, 0, 0, lambda w: loss_fn(W2, w))

    tol = 1e-4
    ok_W2 = abs(dL_dW2[0,0] - ng_W2_00) < tol
    ok_W1 = abs(dL_dW1[0,0] - ng_W1_00) < tol

    print(f"dL/dW2[0,0]: your derivation = {dL_dW2[0,0]:+.6f}, numerical gradient = {ng_W2_00:+.6f}  {'✅' if ok_W2 else '❌'}")
    print(f"dL/dW1[0,0]: your derivation = {dL_dW1[0,0]:+.6f}, numerical gradient = {ng_W1_00:+.6f}  {'✅' if ok_W1 else '❌'}")
    if ok_W2 and ok_W1:
        print("\n✅ Backpropagation derivation checks out")
    else:
        print("\n❌ Something's wrong with the formulas — go back to your derivation and recheck the TODO lines")

## ✏️ Derivation exercise: the gradients of a two-layer network by hand

Given the two-layer network: `loss = ((W2 @ relu(W1 @ x)) - y)²`

**Closed-book task**: without looking at the code above, derive below:
1. `∂loss/∂W2` = ? (hint: let `h = relu(W1 @ x)`, `ŷ = W2 @ h`)
2. `∂loss/∂W1` = ? (hint: the chain rule passes through the relu)

In the code cell below: use numerical differentiation (finite differences) to check whether your derivation is right.

In [ ]:
import numpy as np

# A simple two-layer network (scalar output)
np.random.seed(0)
x  = np.array([1.0, 2.0])
y  = np.array([1.0])
W1 = np.random.randn(3, 2) * 0.5
W2 = np.random.randn(1, 3) * 0.5

def forward(W1, W2, x, y):
    h    = np.maximum(0, W1 @ x)   # relu
    yhat = W2 @ h
    loss = float(((yhat - y) ** 2).sum())
    return loss, h, yhat

loss, h, yhat = forward(W1, W2, x, y)

# ✏️ TODO: replace ... with your hand-derived formulas (hint: ∂loss/∂ŷ = 2(ŷ-y))
dL_dW2 = ...   # same shape as W2 (hint: the outer product of some vector with h)
dL_dW1 = ...   # same shape as W1 (hint: pass back through the relu mask (W1@x > 0), then outer product with x)

# --- Numerical gradient check (central difference; runs automatically once you've filled in your derivation) ---
if dL_dW2 is ... or dL_dW1 is ...:
    print('⬜ TODO: first fill in your hand-derived formulas for dL_dW2 / dL_dW1, then run this cell for the numerical check.')
else:
    eps = 1e-5
    num_grad_W2 = np.zeros_like(W2)
    for i in range(W2.shape[0]):
        for j in range(W2.shape[1]):
            W2p = W2.copy(); W2p[i,j] += eps
            W2m = W2.copy(); W2m[i,j] -= eps
            num_grad_W2[i,j] = (forward(W1,W2p,x,y)[0] - forward(W1,W2m,x,y)[0]) / (2*eps)
    np.testing.assert_allclose(dL_dW2, num_grad_W2, atol=1e-5)
    print("✅ dL/dW2 numerical check passed")

    num_grad_W1 = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            W1p = W1.copy(); W1p[i,j] += eps
            W1m = W1.copy(); W1m[i,j] -= eps
            num_grad_W1[i,j] = (forward(W1p,W2,x,y)[0] - forward(W1m,W2,x,y)[0]) / (2*eps)
    np.testing.assert_allclose(dL_dW1, num_grad_W1, atol=1e-5)
    print("✅ dL/dW1 numerical check passed")

In [ ]:
# ✏️ Self-review for this lesson
l56_review = {
    "topo_order_understood":   None,  # understand why backprop is only correct in reverse topological order? True/False
    "backward_implemented":    None,  # backward() with DFS + reverse order + grad accumulation all implemented correctly? True/False
    "grad_init_understood":    None,  # remember that L.grad=1.0 is the chain rule's starting point, not an arbitrary choice? True/False
    "accumulation_vs_assign":  None,  # the += vs = difference in the diamond-graph demo checked out? True/False
    "whiteboard_passed":       None,  # whiteboard derivation of the backprop matrix formulas passed? True/False
}

unfilled = [k for k, v in l56_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l56_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L56 fully cleared! On to L57: an MLP from scratch')

---

→ **Next lesson** · [L57 · An MLP from scratch](L57_mlp.ipynb)

> Next lesson covers **an MLP from scratch**: hand-written fully connected (FC) layers and activation functions, with the complete forward / backward implementation.